# SaluteAI — โหมดทดสอบ (Test Mode)

ทดสอบท่าวันทยหัตถ์แบบจับเวลา แล้วสรุป **ความแม่นยำรวม** ใน output ของ cell

### ขั้นตอนต่อ 1 ครั้ง
1. หน้าพร้อม → กด **SPACE**
2. นับถอยหลัง **3 – 2 – 1**
3. **ยกมือเมื่อพร้อม** — ระบบเริ่มจับเวลาอัตโนมัติ **ตอนเริ่มยกมือ** แล้วจับต่ออีก **4 วินาที** (ยกมือ → สะบัดหน้า → ค้าง)
   - ไม่ต้องรีบ: จะยกมือวินาทีไหนก็ได้ ภายใน 10 วินาที (ไม่ยกเลย = ไม่ได้ทำท่า)
   - Siamese จะเทียบเฉพาะช่วงตั้งแต่เริ่มยกมือ จึงไม่สนว่าเริ่มช้าหรือเร็ว
4. หน้าผลลัพธ์ของครั้งนั้น → กด **SPACE** ไปครั้งถัดไป

### วิธีคิดคะแนน
| ค่า | ความหมาย |
|---|---|
| **ความแม่นยำ** | เริ่มที่ 100% แล้ว **ผิด 1 อย่าง หัก 10%** (กฎ feedback 10 ข้อ) |
| | Siamese บอกว่าไม่คล้ายต้นแบบ = หักเพิ่มอีก 10% |
| | ไม่ทำท่า / ใช้มือผิดข้าง = 0% |
| **ความคล้ายต้นแบบ** | จาก Siamese (แสดงไว้ดูประกอบ) |
| **ผ่านเกณฑ์ท่าทาง** | ผ่านกฎกี่ข้อจาก 10 ข้อ |
| **หักจาก** | หน้าผลลัพธ์และ output บอกทุกรายการที่หัก เช่น `-10% ส้นเท้าไม่ชิด`, `-10% ท่าโดยรวมไม่คล้ายต้นแบบ` |
| **ส่วนที่ต่างจากต้นแบบ** | ถ้าไม่ผ่าน จะบอกว่าส่วนไหน (ศีรษะ / แขน / สะโพก / ขา) ต่างจากท่าต้นแบบมากที่สุด |
| **ผ่าน / ไม่ผ่าน** | ผ่าน = Siamese บอกว่าคล้ายต้นแบบ **และ** ไม่ผิดกฎข้อไหนเลย |

**ความแม่นยำรวม** = ค่าเฉลี่ยความแม่นยำของทุกครั้งที่ทดสอบ

### เพิ่ม dataset ท่าถูก
ถ้าครั้งไหน **ทำถูกจริง** ให้กด **C** ในหน้าผลลัพธ์ → ครั้งนั้นถูกบันทึกเข้า `dataset/correct` (ใช้ได้เฉพาะครั้งที่ไม่ผิดกฎ) แล้วรัน cell 2 → 3 → 4 ใน `siamese_learn` เพื่อเทรนใหม่ — ท่าถูกจากหลายวัน/หลายตำแหน่งช่วยให้ Siamese แม่นและนิ่งขึ้นมาก

### ก่อนใช้
- ต้องมี `models/encoder.keras`, `models/threshold.txt` (cell 4 ใน `siamese_learn`) และ `models/rules.json` (cell 4.5)
- รัน cell **1. Setup** ครั้งเดียว แล้วรัน cell **2. Run test** กี่รอบก็ได้
- ระหว่างทดสอบ กด **Q** เพื่อหยุด (สรุปผลเฉพาะครั้งที่ทำเสร็จแล้ว)

# 1. Setup (รันครั้งเดียว)

In [15]:
import cv2
import json
import time
import importlib
from collections import deque
from datetime import datetime
from pathlib import Path

import mediapipe as mp
import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont

import salute_rules as rules_lib
importlib.reload(rules_lib)

# -------------------------
# Configuration
# -------------------------
NUM_ATTEMPTS = 3              # จำนวนครั้งที่ทดสอบในหนึ่งรอบ
COUNTDOWN_SECONDS = 3         # นับถอยหลังก่อนจับเวลา
CLIP_SECONDS = 6              # ความยาวคลิปที่ประเมิน (ต้องเท่ากับตอนอัด)
PRE_ONSET_SECONDS = 2.0       # เก็บก่อนเริ่มยกมือ (ยืนตรง)
POST_ONSET_SECONDS = CLIP_SECONDS - PRE_ONSET_SECONDS   # จับเวลาหลังเริ่มยกมือ = 4 วิ
WAIT_TIMEOUT = 10             # ไม่ยกมือภายในกี่วินาที = ไม่ได้ทำท่า
MIN_IN_FRAME = 0.8            # หัว/เท้าต้องอยู่ในภาพอย่างน้อย 80% ของเวลา ไม่งั้นเตือนว่าผลอาจไม่แม่น
SAMPLE_FPS = 10
SEQUENCE_LENGTH = CLIP_SECONDS * SAMPLE_FPS   # = 60 เฟรม
FEATURES = 218
CAMERA_ID = 0
SAVE_RESULTS = True           # บันทึกผลเป็นไฟล์ JSON ใน results/
PENALTY_PER_ERROR = 10        # ผิด 1 อย่าง หักความแม่นยำกี่ %
SIAMESE_PENALTY = 10          # ท่าไม่คล้ายต้นแบบ (Siamese) นับเป็นผิดอีก 1 อย่าง (0 = ไม่หัก)

CORRECT_DIR = Path("dataset/correct")
ENCODER_PATH = Path("models/encoder.keras")
THRESHOLD_PATH = Path("models/threshold.txt")
RULES_PATH = Path("models/rules.json")
RESULTS_DIR = Path("results")

for path in [ENCODER_PATH, THRESHOLD_PATH, RULES_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"ไม่พบ {path} - รัน cell 4 (Evaluate) และ 4.5 (Calibrate) ใน siamese_learn ก่อน"
        )

threshold = float(THRESHOLD_PATH.read_text().strip())
feedback_rules = rules_lib.load_rules(RULES_PATH)

encoder = tf.keras.models.load_model(ENCODER_PATH, compile=False)

# ท่าต้นแบบ (ท่าถูก) → embedding เก็บไว้ครั้งเดียว
references = []

for folder in sorted(
    (d for d in CORRECT_DIR.iterdir() if d.is_dir() and d.name.isdigit()),
    key=lambda d: int(d.name)
):
    try:
        clip = rules_lib.load_clip(folder, SEQUENCE_LENGTH)
    except (FileNotFoundError, ValueError):
        continue

    if clip.shape == (SEQUENCE_LENGTH, FEATURES):
        references.append(clip)

if not references:
    raise RuntimeError("ไม่พบท่าต้นแบบใน dataset/correct")

# จัดจังหวะให้เริ่มตอนยกมือ เหมือนตอนเทรน (ตัดคลิปที่หาจุดเริ่มยกมือไม่เจอออก)
references = [rules_lib.align_sequence(s, fallback_to_start=False) for s in references]
references = [s for s in references if s is not None]

# Siamese ไม่ใช้ค่า z (ความลึก) → เลือกเฉพาะคอลัมน์ SIAMESE_COLUMNS
reference_embeddings = encoder.predict(
    np.stack(references)[:, :, rules_lib.SIAMESE_COLUMNS], verbose=0
)


# -------------------------
# MediaPipe + feature extraction (เหมือนตอนอัด dataset)
# -------------------------
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

WINDOW_NAME = "SaluteAI - Test Mode"


# -------------------------
# 2. Extract 218 features
# -------------------------
# Pose joints ที่ใช้ (index ของ MediaPipe Pose)
POSE_JOINTS = {
    "face": list(range(0, 11)),   # จมูก ตา หู ปาก (11 จุด)
    "shoulders": [11, 12],
    "elbows": [13, 14],
    "wrists": [15, 16],
    "hips": [23, 24],             # สะโพก (ต้นขา)
    "knees": [25, 26],            # เข่า
    "ankles": [27, 28],           # ข้อเท้า
}
POSE_INDEXES = [i for group in POSE_JOINTS.values() for i in group]

HAND_POINTS = 21                  # จุดบนมือข้างละ 21 จุด (นิ้วครบ)

# 23 pose x (x, y, z, visibility) + 2 มือ x 21 x (x, y, z) = 92 + 126 = 218
FEATURES = len(POSE_INDEXES) * 4 + 2 * HAND_POINTS * 3


def extract_landmarks(results):
    if results.pose_landmarks is None:
        return None

    landmarks = results.pose_landmarks.landmark

    # Shoulder landmarks
    left_shoulder = landmarks[11]
    right_shoulder = landmarks[12]

    center_x = (
        left_shoulder.x + right_shoulder.x
    ) / 2

    center_y = (
        left_shoulder.y + right_shoulder.y
    ) / 2

    scale = np.sqrt(
        (left_shoulder.x - right_shoulder.x) ** 2
        + (left_shoulder.y - right_shoulder.y) ** 2
    )

    scale = max(scale, 1e-6)

    values = []

    # Pose: หน้า ไหล่ ศอก ข้อมือ สะโพก เข่า ข้อเท้า
    for index in POSE_INDEXES:
        landmark = landmarks[index]

        # Normalize coordinates relative to shoulders
        x = (landmark.x - center_x) / scale
        y = (landmark.y - center_y) / scale
        z = landmark.z / scale
        visibility = landmark.visibility

        values.extend([x, y, z, visibility])

    # Hands: ถ้าตรวจไม่เจอมือข้างไหน ใส่ 0 แทน (เหมือน LSTM-learn)
    for hand in [
        results.left_hand_landmarks,
        results.right_hand_landmarks
    ]:
        if hand is None:
            values.extend([0.0] * HAND_POINTS * 3)
            continue

        for landmark in hand.landmark:
            x = (landmark.x - center_x) / scale
            y = (landmark.y - center_y) / scale
            z = landmark.z / scale

            values.extend([x, y, z])

    return np.array(values, dtype=np.float32)


def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image,
        results.pose_landmarks,
        mp_holistic.POSE_CONNECTIONS
    )
    mp_drawing.draw_landmarks(
        image,
        results.left_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS
    )
    mp_drawing.draw_landmarks(
        image,
        results.right_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS
    )


# -------------------------
# Drawing helpers
# -------------------------
def put_text(image, text, org, font, scale, color, thickness=2, line_type=cv2.LINE_AA):
    # ตัวหนังสือหนา มีขอบดำ อ่านชัดทุกพื้นหลัง
    thickness += 1
    cv2.putText(image, text, org, font, scale, (0, 0, 0), thickness + 4, line_type)
    cv2.putText(image, text, org, font, scale, color, thickness, line_type)


# -------------------------
# Thai text (cv2 เขียนภาษาไทยไม่ได้ ใช้ Pillow + ฟอนต์ไทยแทน)
# -------------------------
THAI_FONT_CANDIDATES = [
    Path("../../KanitFont/Kanit-Bold.ttf"),
    Path("C:/Windows/Fonts/Kanit-Medium.ttf"),
    Path("C:/Windows/Fonts/tahomabd.ttf"),
    Path("C:/Windows/Fonts/LeelawUI.ttf"),
]
_thai_fonts = {}


def thai_font(size):
    if size not in _thai_fonts:
        path = next((p for p in THAI_FONT_CANDIDATES if p.exists()), None)

        if path is None:
            raise FileNotFoundError("ไม่พบฟอนต์ภาษาไทย")

        _thai_fonts[size] = ImageFont.truetype(str(path), size)

    return _thai_fonts[size]


def wrap_thai(draw, text, font, max_width):
    # ตัดบรรทัดตรงช่องว่าง เมื่อข้อความยาวเกินจอ
    lines, line = [], ""

    for word in text.split(" "):
        candidate = f"{line} {word}".strip()

        if line and draw.textlength(candidate, font=font) > max_width:
            lines.append(line)
            line = word
        else:
            line = candidate

    lines.append(line)
    return lines


def draw_thai(image, items):
    # items: [(text, (x, y), size, (B, G, R)), ...] ตัวหนังสือขอบดำ หนา อ่านชัด
    pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)

    for text, (x, y), size, color in items:
        font = thai_font(size)

        for line in wrap_thai(draw, text, font, image.shape[1] - x - 15):
            draw.text(
                (x, y), line, font=font,
                fill=tuple(color[::-1]),
                stroke_width=max(2, size // 10),
                stroke_fill=(0, 0, 0)
            )
            y += int(size * 1.35)

    image[:] = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)


def draw_header(image, text, color):
    cv2.rectangle(image, (0, 0), (image.shape[1], 40), color, -1)
    put_text(
        image, text, (10, 28),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7, (255, 255, 255), 2, cv2.LINE_AA
    )


def resample_sequence(times, frames):
    # เลือกเฟรมที่ใกล้เวลาเป้าหมายที่สุด ให้ได้ SEQUENCE_LENGTH เฟรมเท่ากันทุกคลิป
    # ไม่ว่ากล้อง + MediaPipe จะได้กี่ FPS
    times = np.asarray(times, dtype=np.float64)
    times = times - times[0]
    targets = np.linspace(0.0, times[-1], SEQUENCE_LENGTH)
    indexes = np.abs(times[None, :] - targets[:, None]).argmin(axis=1)

    return [frames[i] for i in indexes]


def read_frame(cap, pose):
    success, frame = cap.read()

    if not success:
        return None, None

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    draw_landmarks(frame, results)

    return frame, results


def darken(image, alpha, top=0, bottom=None):
    bottom = image.shape[0] if bottom is None else bottom
    region = image[top:bottom]
    image[top:bottom] = (region * (1 - alpha)).astype(np.uint8)


def show(frame):
    cv2.imshow(WINDOW_NAME, frame)
    return cv2.waitKey(1) & 0xFF


# -------------------------
# Scoring
# -------------------------
TOTAL_RULES = len(rules_lib.PRIORITY)


# ส่วนของร่างกาย (สำหรับบอกว่า "ต่างจากต้นแบบตรงไหน")
BODY_PARTS = {
    "ศีรษะ/หน้า": list(range(0, 11)),
    "ไหล่และแขน": [11, 12, 13, 14, 15, 16],
    "สะโพก": [23, 24],
    "ขาและเท้า": [25, 26, 27, 28],
}
PART_COLUMNS = {
    name: [
        c for index in indexes
        for c in (4 * rules_lib.POSE_INDEXES.index(index), 4 * rules_lib.POSE_INDEXES.index(index) + 1)
    ]
    for name, indexes in BODY_PARTS.items()
}
reference_stack = np.stack(references)
reference_mean = reference_stack.mean(axis=0)
reference_std = reference_stack.std(axis=0) + 0.05


def part_differences(aligned):
    """ส่วนไหนต่างจากท่าต้นแบบกี่เท่าของความต่างปกติระหว่างท่าต้นแบบด้วยกัน (1 = ปกติ)"""
    return {
        name: round(float(np.mean(
            np.abs(aligned[:, cols] - reference_mean[:, cols]) / reference_std[:, cols]
        )), 2)
        for name, cols in PART_COLUMNS.items()
    }


def score_attempt(sequence, framing=None):
    # Siamese เทียบเฉพาะช่วงตั้งแต่เริ่มยกมือ (ไม่สนว่ายกมือวินาทีไหน)
    aligned = rules_lib.align_sequence(sequence)
    embedding = encoder.predict(
        aligned[np.newaxis, :, rules_lib.SIAMESE_COLUMNS], verbose=0
    )
    distances = np.linalg.norm(reference_embeddings - embedding, axis=1)

    # ระยะเฉลี่ยกับท่าต้นแบบที่ใกล้ที่สุด 3 คลิป (เหมือน cell Test เดิม)
    distance = float(np.mean(np.sort(distances)[:3]))
    similarity = 100.0 * float(np.clip(1.0 - distance / (2.0 * threshold), 0.0, 1.0))
    is_similar = distance <= threshold

    errors = rules_lib.detect_errors(sequence, feedback_rules)
    parts = part_differences(aligned)

    # รายการหักคะแนน: (เหตุผล, %)
    deductions = []

    if errors and errors[0] in ("no_salute", "wrong_hand"):
        # ไม่ได้ทำท่า / ใช้มือผิดข้าง = ทำผิดทั้งท่า ได้ 0
        rules_passed = 0
        deductions.append((rules_lib.FEEDBACK_TH[errors[0]][0], 100))
    else:
        rules_passed = TOTAL_RULES - len(errors)

        # ผิด 1 อย่าง หัก PENALTY_PER_ERROR
        for code in errors:
            deductions.append((rules_lib.FEEDBACK_TH[code][0], PENALTY_PER_ERROR))

        if not is_similar and SIAMESE_PENALTY:
            deductions.append(("ท่าโดยรวมไม่คล้ายต้นแบบ", SIAMESE_PENALTY))

    penalty = min(100, sum(amount for _, amount in deductions))
    accuracy = max(0.0, 100.0 - penalty)

    # อยู่ในภาพครบไหม (หัวหลุดบน / เท้าหลุดล่าง = MediaPipe เดาจุด ผลอาจไม่แม่น)
    framing = framing or {"head": 1.0, "feet": 1.0}
    body_visible = framing["head"] >= MIN_IN_FRAME and framing["feet"] >= MIN_IN_FRAME

    return {
        "distance": round(distance, 4),
        "similarity": round(similarity, 1),
        "rules_passed": rules_passed,
        "penalty": penalty,
        "deductions": deductions,
        "accuracy": round(accuracy, 1),
        "passed": bool(is_similar and not errors),
        "errors": errors,
        "part_diff": parts,
        "most_different": max(parts, key=parts.get),
        "framing": framing,
        "body_visible": bool(body_visible),
    }


def in_frame(results):
    """หัว (จมูก) และเท้า (ข้อเท้า) อยู่ในภาพจริงไหม (ดูจากพิกัด ไม่ใช่ค่า visibility)"""
    if results.pose_landmarks is None:
        return False, False

    lm = results.pose_landmarks.landmark
    inside = lambda p: 0.0 <= p.x <= 1.0 and 0.0 <= p.y <= 1.0

    return inside(lm[0]), inside(lm[27]) and inside(lm[28])


# -------------------------
# Screens
# -------------------------
def screen_ready(cap, pose, attempt):
    # หน้าพร้อม: รอกด SPACE
    while True:
        frame, results = read_frame(cap, pose)

        if frame is None:
            continue

        darken(frame, 0.5)
        items = [
            (f"ครั้งที่ {attempt}/{NUM_ATTEMPTS}", (25, 30), 30, (200, 200, 200)),
            ("โหมดทดสอบ ท่าวันทยหัตถ์", (25, 80), 40, (255, 255, 255)),
            (f"เมื่อพร้อม กด SPACE แล้วนับถอยหลัง {COUNTDOWN_SECONDS} วินาที", (25, 150), 22, (255, 255, 255)),
            ("จากนั้นยกมือเมื่อพร้อม ระบบเริ่มจับเวลาตอนเริ่มยกมือ", (25, 185), 22, (255, 255, 255)),
            ("ยืนให้เห็นทั้งตัว ตั้งแต่หัวถึงเท้า", (25, 220), 22, (200, 255, 200)),
            ("SPACE = เริ่ม     Q = ออก", (25, frame.shape[0] - 55), 26, (255, 255, 255)),
        ]

        head_in, feet_in = in_frame(results)

        if results.pose_landmarks is None:
            items.append(("ตรวจไม่เจอตัว - ถอยให้เห็นทั้งตัว", (25, 270), 24, (80, 80, 255)))
        elif not (head_in and feet_in):
            missing = "หัวและเท้า" if not (head_in or feet_in) else ("หัว" if not head_in else "เท้า")
            items.append((f"{missing}หลุดเฟรม - ถอยห่างกล้องให้เห็นทั้งตัว", (25, 270), 24, (80, 80, 255)))
        else:
            items.append(("เห็นทั้งตัวแล้ว พร้อมทดสอบ", (25, 270), 24, (0, 230, 0)))

        draw_thai(frame, items)
        key = show(frame)

        if key == ord(" "):
            return True

        if key == ord("q"):
            return False


def screen_countdown(cap, pose):
    start = time.time()

    while time.time() - start < COUNTDOWN_SECONDS:
        frame, _ = read_frame(cap, pose)

        if frame is None:
            continue

        remaining = COUNTDOWN_SECONDS - int(time.time() - start)

        draw_thai(frame, [
            ("เตรียมตัว", (frame.shape[1] // 2 - 70, 60), 36, (255, 255, 255)),
            (str(remaining), (frame.shape[1] // 2 - 40, 150), 140, (255, 255, 255)),
        ])

        if show(frame) == ord("q"):
            return False

    return True


def resample_fixed(times, frames, end_time):
    # เลือกเฟรมทุก 0.1 วิ ย้อนหลัง CLIP_SECONDS จาก end_time (เวลาจริง ไม่ยืด/หด)
    times = np.asarray(times, dtype=np.float64)
    targets = np.linspace(end_time - CLIP_SECONDS, end_time, SEQUENCE_LENGTH)
    indexes = np.abs(times[None, :] - targets[:, None]).argmin(axis=1)

    return [frames[i] for i in indexes]


LAST_FRAMING = {}   # สัดส่วนเวลาที่หัว/เท้าอยู่ในภาพ ของการจับเวลาครั้งล่าสุด


def screen_record(cap, pose):
    """
    รอจนเริ่มยกมือ → จับเวลา POST_ONSET_SECONDS วินาที
    คืน sequence (60, 218) = ก่อนยกมือ PRE_ONSET_SECONDS + หลังยกมือ POST_ONSET_SECONDS
    หรือ "retry" / None (กด Q)
    """
    buffer = deque()            # (time, features)
    onset_time = None
    wait_start = time.time()
    framing_counts = [0, 0, 0]  # frames, head in frame, feet in frame

    while True:
        frame, results = read_frame(cap, pose)

        if frame is None:
            continue

        now = time.time()
        features = extract_landmarks(results)
        items = []

        if onset_time is not None:
            head_in, feet_in = in_frame(results)
            framing_counts[0] += 1
            framing_counts[1] += head_in
            framing_counts[2] += feet_in

        if features is None:
            # ตัวหลุดเฟรม → เริ่มรอใหม่
            buffer.clear()
            onset_time = None
            items.append(("ตรวจไม่เจอตัว - ถอยให้เห็นทั้งตัว", (15, 60), 26, (80, 80, 255)))
        else:
            buffer.append((now, features))

        if onset_time is None:
            # ระหว่างรอ เก็บย้อนหลังแค่ CLIP_SECONDS
            while buffer and now - buffer[0][0] > CLIP_SECONDS:
                buffer.popleft()

            # ระดับมือตอนยืนตรง = ช่วงต้นของ PRE_ONSET_SECONDS ล่าสุด
            recent = [f for t, f in buffer if now - t <= PRE_ONSET_SECONDS]

            if len(recent) >= 5:
                heights = rules_lib.wrist_height(np.array(recent))
                base = float(np.median(heights[:5]))

                if base >= rules_lib.HAND_DOWN_Y and base - heights[-1] > rules_lib.ONSET_RISE:
                    onset_time = now
                elif heights[-1] < rules_lib.HAND_DOWN_Y and base < rules_lib.HAND_DOWN_Y:
                    items.append(("ลดมือลงข้างลำตัวก่อน แล้วค่อยยกมือ", (15, 60), 26, (80, 80, 255)))

            remaining = WAIT_TIMEOUT - (now - wait_start)

            if remaining <= 0:
                break   # ไม่ยกมือภายในเวลา → ประเมินช่วงล่าสุด (จะได้ "ไม่ได้ทำท่า")

        if onset_time is None:
            cv2.rectangle(frame, (0, 0), (frame.shape[1], 45), (120, 70, 0), -1)
            items.append(("ยกมือทำวันทยหัตถ์เมื่อพร้อม", (15, 5), 28, (255, 255, 255)))
            items.append(
                (f"เริ่มจับเวลาอัตโนมัติเมื่อยกมือ (รอได้อีก {max(remaining, 0):.0f} วิ)",
                 (15, frame.shape[0] - 45), 20, (255, 255, 255))
            )
        else:
            elapsed = now - onset_time

            cv2.rectangle(frame, (0, 0), (frame.shape[1], 45), (0, 0, 200), -1)
            items.append(
                (f"กำลังจับเวลา  {elapsed:.1f} / {POST_ONSET_SECONDS:.0f} วินาที", (15, 5), 28, (255, 255, 255))
            )

            progress = int(frame.shape[1] * min(elapsed / POST_ONSET_SECONDS, 1.0))
            cv2.rectangle(frame, (0, frame.shape[0] - 12), (progress, frame.shape[0]), (0, 255, 0), -1)

            if elapsed >= POST_ONSET_SECONDS:
                break

        draw_thai(frame, items)

        if show(frame) == ord("q"):
            return None

    if len(buffer) < SEQUENCE_LENGTH // 3:
        print(f"เก็บเฟรมได้น้อยเกินไป ({len(buffer)} เฟรม) - ทดสอบครั้งนี้ใหม่")
        return "retry"

    total = max(framing_counts[0], 1)
    LAST_FRAMING.clear()
    LAST_FRAMING.update({
        "head": round(framing_counts[1] / total, 2) if framing_counts[0] else 1.0,
        "feet": round(framing_counts[2] / total, 2) if framing_counts[0] else 1.0,
    })

    times, frames = zip(*buffer)
    return np.array(resample_fixed(times, frames, times[-1]), dtype=np.float32)


def save_as_correct(sequence):
    """บันทึกครั้งนี้เป็นท่าถูกใน dataset/correct (รูปแบบเดียวกับตอนอัด: 60 ไฟล์ .npy)"""
    indexes = [int(d.name) for d in CORRECT_DIR.iterdir() if d.is_dir() and d.name.isdigit()]
    folder = CORRECT_DIR / str(max(indexes) + 1 if indexes else 0)
    folder.mkdir(parents=True)

    for frame_num, keypoints in enumerate(sequence):
        np.save(folder / f"{frame_num}.npy", keypoints)

    return folder


def screen_result(cap, pose, result, attempt, sequence=None):
    saved_to = None
    can_save = sequence is not None and not result["errors"]

    color = (0, 230, 0) if result["passed"] else (0, 165, 255)

    while True:
        frame, _ = read_frame(cap, pose)

        if frame is None:
            continue

        darken(frame, 0.7)
        items = [
            (f"ผลครั้งที่ {attempt}/{NUM_ATTEMPTS}", (25, 20), 26, (200, 200, 200)),
            (f"ความแม่นยำ {result['accuracy']:.1f}%", (25, 60), 48, color),
            ("ผ่าน" if result["passed"] else "ไม่ผ่าน", (frame.shape[1] - 150, 70), 40, color),
            (f"ความคล้ายต้นแบบ  {result['similarity']:.1f}%", (25, 135), 24, (255, 255, 255)),
            (f"ผ่านเกณฑ์ท่าทาง  {result['rules_passed']}/{TOTAL_RULES} ข้อ", (25, 170), 24, (255, 255, 255)),
        ]

        y = 210

        # หักคะแนนจากอะไรบ้าง
        if result["deductions"]:
            items.append((f"หัก {result['penalty']}%", (frame.shape[1] - 150, 135), 26, (80, 80, 255)))

            for reason, amount in result["deductions"][:4]:
                items.append((f"-{amount}%  {reason}", (25, y), 21, (120, 120, 255)))
                y += 28

            y += 8

        if result["errors"]:
            for code in result["errors"][:3]:
                name, tip = rules_lib.FEEDBACK_TH[code]
                items.append((f"แก้: {tip}", (25, y), 19, (255, 255, 255)))
                y += 28
        elif result["passed"]:
            items.append(("ท่าถูกต้องครบทุกข้อ", (25, y), 28, (0, 230, 0)))
            y += 40
        else:
            # ผ่านกฎทุกข้อ แต่ Siamese บอกว่าไม่คล้าย → บอกว่าต่างตรงไหน
            items.append(("ผ่านกฎทุกข้อ แต่ท่าโดยรวมยังไม่เหมือนต้นแบบ", (25, y), 21, (255, 255, 255)))
            y += 30
            items.append(
                (f"ส่วนที่ต่างจากต้นแบบมากที่สุด: {result['most_different']}", (25, y), 21, (0, 165, 255))
            )
            y += 36

        if not result["body_visible"]:
            items.append(
                ("หัวหรือเท้าหลุดเฟรม ผลอาจไม่แม่น - ถอยให้เห็นทั้งตัวเหมือนตอนอัด",
                 (25, max(y, 330)), 20, (80, 80, 255))
            )

        next_text = "SPACE = ครั้งถัดไป" if attempt < NUM_ATTEMPTS else "SPACE = ดูสรุปผล"
        items.append((f"{next_text}     Q = ออก", (25, frame.shape[0] - 50), 24, (255, 255, 255)))

        # เพิ่ม dataset ท่าถูก: ครั้งที่ทำถูกจริง กด C เก็บเข้า dataset/correct
        if saved_to is not None:
            items.append((f"บันทึกเป็นท่าถูกแล้ว: {saved_to}", (25, frame.shape[0] - 90), 20, (0, 230, 0)))
        elif can_save:
            items.append(("C = บันทึกครั้งนี้เป็นท่าถูก (เพิ่ม dataset)", (25, frame.shape[0] - 90), 20, (200, 255, 200)))

        draw_thai(frame, items)
        key = show(frame)

        if key == ord("c") and can_save and saved_to is None:
            saved_to = save_as_correct(sequence)
            result["saved_as_correct"] = str(saved_to)
            print(f"ครั้งที่ {attempt}: บันทึกเป็นท่าถูก → {saved_to}")

        if key == ord(" "):
            return True

        if key == ord("q"):
            return False


print("พร้อมทดสอบ")
print(f"  ท่าต้นแบบ {len(references)} คลิป | threshold {threshold:.4f}")
print(f"  ทดสอบ {NUM_ATTEMPTS} ครั้ง | เริ่มจับเวลาตอนยกมือ แล้วจับต่อ {POST_ONSET_SECONDS:.0f} วินาที")

พร้อมทดสอบ
  ท่าต้นแบบ 29 คลิป | threshold 0.5281
  ทดสอบ 3 ครั้ง | เริ่มจับเวลาตอนยกมือ แล้วจับต่อ 4 วินาที


# 2. Run test (รันซ้ำได้ทุกครั้งที่ต้องการทดสอบ)

In [17]:
attempt_results = []
attempt_sequences = []

cap = cv2.VideoCapture(CAMERA_ID)

if not cap.isOpened():
    raise RuntimeError("เปิดกล้องไม่ได้ กรุณาตรวจสอบ CAMERA_ID")

try:
    with mp_holistic.Holistic(
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as pose:

        attempt = 1

        while attempt <= NUM_ATTEMPTS:
            if not screen_ready(cap, pose, attempt):
                break

            if not screen_countdown(cap, pose):
                break

            sequence = screen_record(cap, pose)

            if sequence is None:
                break

            if isinstance(sequence, str):   # "retry"
                continue

            result = score_attempt(sequence, dict(LAST_FRAMING))
            result["attempt"] = attempt
            attempt_results.append(result)
            attempt_sequences.append(sequence)

            if not screen_result(cap, pose, result, attempt, sequence):
                break

            attempt += 1

finally:
    cap.release()
    cv2.destroyAllWindows()


# -------------------------
# สรุปผล (output ของ cell)
# -------------------------
if not attempt_results:
    print("ยังไม่มีผลการทดสอบ")
else:
    print("=" * 60)
    print("ผลการทดสอบท่าวันทยหัตถ์")
    print("=" * 60)

    for r in attempt_results:
        errors = ", ".join(rules_lib.FEEDBACK_TH[c][0] for c in r["errors"]) or "-"
        print(
            f"ครั้งที่ {r['attempt']}: "
            f"ความแม่นยำ {r['accuracy']:5.1f}%  "
            f"(คล้ายต้นแบบ {r['similarity']:5.1f}% | "
            f"ท่าทาง {r['rules_passed']}/{TOTAL_RULES} | หัก {r['penalty']}%)  "
            f"{'ผ่าน' if r['passed'] else 'ไม่ผ่าน'}"
        )
        print(f"           จุดที่ผิด: {errors}")

        if r["deductions"]:
            reasons = ", ".join(f"{reason} -{amount}%" for reason, amount in r["deductions"])
            print(f"           หักจาก: {reasons}")

        if not r["passed"]:
            parts = ", ".join(f"{name} {value:.1f}" for name, value in r["part_diff"].items())
            print(f"           ต่างจากต้นแบบ (1 = ปกติ): {parts}")

        if not r["body_visible"]:
            print(
                f"           (หัวอยู่ในภาพ {r['framing']['head']:.0%} | "
                f"เท้าอยู่ในภาพ {r['framing']['feet']:.0%} ของเวลา → ผลครั้งนี้อาจไม่แม่น)"
            )

    accuracies = [r["accuracy"] for r in attempt_results]
    passed = sum(r["passed"] for r in attempt_results)

    print("-" * 60)
    print(f"ความแม่นยำรวม (เฉลี่ย {len(attempt_results)} ครั้ง): {np.mean(accuracies):.1f}%")
    print(f"สูงสุด {max(accuracies):.1f}% | ต่ำสุด {min(accuracies):.1f}%")
    print(f"ผ่าน {passed}/{len(attempt_results)} ครั้ง")

    error_counts = {}

    for r in attempt_results:
        for code in r["errors"]:
            error_counts[code] = error_counts.get(code, 0) + 1

    if error_counts:
        print("\nจุดที่ผิดบ่อย:")

        for code, count in sorted(error_counts.items(), key=lambda item: -item[1]):
            name, tip = rules_lib.FEEDBACK_TH[code]
            print(f"  - {name} ({count} ครั้ง) → {tip}")

    if SAVE_RESULTS:
        RESULTS_DIR.mkdir(exist_ok=True)
        stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
        out_path = RESULTS_DIR / f"test_{stamp}.json"

        # เก็บข้อมูลท่าแต่ละครั้งไว้วิเคราะห์ภายหลัง
        np.save(RESULTS_DIR / f"test_{stamp}_sequences.npy", np.stack(attempt_sequences))
        out_path.write_text(
            json.dumps({
                "overall_accuracy": round(float(np.mean(accuracies)), 1),
                "passed": passed,
                "attempts": attempt_results,
            }, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )
        print("\nบันทึกผล:", out_path)

ผลการทดสอบท่าวันทยหัตถ์
ครั้งที่ 1: ความแม่นยำ 100.0%  (คล้ายต้นแบบ  84.9% | ท่าทาง 10/10 | หัก 0%)  ผ่าน
           จุดที่ผิด: -
ครั้งที่ 2: ความแม่นยำ 100.0%  (คล้ายต้นแบบ  92.1% | ท่าทาง 10/10 | หัก 0%)  ผ่าน
           จุดที่ผิด: -
ครั้งที่ 3: ความแม่นยำ  90.0%  (คล้ายต้นแบบ  51.4% | ท่าทาง 9/10 | หัก 10%)  ไม่ผ่าน
           จุดที่ผิด: ไม่สะบัดหน้า
           หักจาก: ไม่สะบัดหน้า -10%
           ต่างจากต้นแบบ (1 = ปกติ): ศีรษะ/หน้า 0.8, ไหล่และแขน 0.4, สะโพก 0.8, ขาและเท้า 0.7
------------------------------------------------------------
ความแม่นยำรวม (เฉลี่ย 3 ครั้ง): 96.7%
สูงสุด 100.0% | ต่ำสุด 90.0%
ผ่าน 2/3 ครั้ง

จุดที่ผิดบ่อย:
  - ไม่สะบัดหน้า (1 ครั้ง) → ยกอกอึ้บ แล้วสะบัดหน้าไปทางซ้าย

บันทึกผล: results\test_20261011_012514.json
